# 01 — Amostra sistemática e séries Sentinel-2 via WTSS

Este caderno documenta e audita os dados que entram no Pixel Purity Index (PPI). O código está integralmente exposto: não há importação de módulos locais do projeto.

O fluxo tem quatro objetivos:

1. ler a configuração congelada do estudo;
2. verificar os 1.000 pixels sistemáticos de cada classe;
3. verificar as 184 observações temporais esperadas por pixel;
4. mostrar, de forma opcional, como uma série pode ser novamente consultada no serviço WTSS do Brasil Data Cube.

A consulta remota permanece desativada por padrão. Os arquivos Parquet distribuídos em `data/wtss/` correspondem exatamente às séries utilizadas no estudo e bastam para reproduzir o PPI.

## Dependências e localização dos arquivos

São utilizadas apenas bibliotecas científicas de uso geral. A função `localizar_repositorio` percorre os diretórios ascendentes até encontrar `config/study.yaml`, evitando caminhos absolutos e permitindo executar o caderno em Windows, Linux ou macOS.

In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import yaml


def localizar_repositorio(inicio=None):
    """Localiza a raiz sem pressupor o sistema operacional ou caminho local."""
    atual = Path(inicio or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "config" / "study.yaml").is_file() and (candidato / "data").is_dir():
            return candidato
    raise FileNotFoundError("A raiz do repositório não foi encontrada.")


RAIZ = localizar_repositorio()
with (RAIZ / "config" / "study.yaml").open(encoding="utf-8") as arquivo:
    CONFIG = yaml.safe_load(arquivo)

CLASSES = {str(codigo).zfill(2): nome for codigo, nome in CONFIG["classes"].items()}
BANDAS = list(CONFIG["bands"])
SAIDA = RAIZ / "outputs" / "standalone"
SAIDA.mkdir(parents=True, exist_ok=True)

print(f"Repositório: {RAIZ}")
print(f"Classes: {', '.join(CLASSES)}")
print(f"Bandas: {', '.join(BANDAS)}")

## Delineamento amostral

Cada classe TerraClass constitui um estrato temático. Dentro de cada estrato foi construída uma malha sistemática com origem pseudoaleatória reproduzível. O espaçamento inicial é dado por

\[
h_c^{(0)}=\sqrt{\frac{\widetilde A_c}{n}},
\]

em que \(\widetilde A_c\) é a área operacional do núcleo permanente da classe \(c\) e \(n=1.000\) é o limite igual adotado para todas as classes. Se a malha não alcança \(n\) pixels elegíveis, o espaçamento é reduzido em passos de 5%:

\[
h_c^{(j)}=0{,}95^j h_c^{(0)}.
\]

A origem é determinada por dois números uniformes produzidos com seed 13. Quando a malha contém mais de 1.000 pixels, os candidatos são ordenados espacialmente pelo código de Morton e reduzidos em intervalos regulares. Os 1.000 pontos são um limite operacional, e não um tamanho amostral inferencial nem uma garantia de independência espacial.

In [ ]:
pontos = pd.read_parquet(RAIZ / "data" / "points" / "points.parquet")
pontos["class_code"] = pontos["class_code"].astype(str).str.zfill(2)

esperados = int(CONFIG["sampling"]["points_per_class"])
contagens = pontos.groupby("class_code").size().reindex(CLASSES).fillna(0).astype(int)
duplicados = pontos.duplicated(["class_code", "source_row", "source_col"]).sum()
seeds = sorted(pd.to_numeric(pontos["sampling_seed"], errors="raise").unique().tolist())

assert len(pontos) == esperados * len(CLASSES)
assert (contagens == esperados).all()
assert duplicados == 0
assert seeds == [int(CONFIG["sampling"]["seed"])]

resumo_pontos = (
    pontos.groupby(["class_code", "class_name"], as_index=False)
    .agg(
        pontos=("point_id", "size"),
        espacamento_nominal_m=("nominal_spacing_m", "first"),
        espacamento_final_m=("final_spacing_m", "first"),
        distancia_minima_m=("class_min_distance_m", "first"),
    )
)
resumo_pontos

### Distâncias observadas entre pontos

A distância ao vizinho mais próximo de cada ponto é calculada no sistema projetado SIRGAS 2000/Brazil Polyconic (EPSG:5880):

\[
d_i=\min_{j\ne i}\lVert s_i-s_j\rVert_2.
\]

O cálculo abaixo utiliza uma busca em árvore k-d. O primeiro vizinho retornado é o próprio ponto; por isso, a segunda distância é a distância ao vizinho distinto mais próximo.

In [ ]:
try:
    from scipy.spatial import cKDTree
except ImportError as erro:
    raise ImportError("Instale scipy para calcular as distâncias entre pontos.") from erro

linhas_distancia = []
for codigo, grupo in pontos.groupby("class_code", sort=True):
    coordenadas = grupo[["x_5880", "y_5880"]].to_numpy(float)
    distancias, _ = cKDTree(coordenadas).query(coordenadas, k=2)
    vizinho = distancias[:, 1]
    linhas_distancia.append(
        {
            "class_code": codigo,
            "n": len(grupo),
            "media_m": float(vizinho.mean()),
            "mediana_m": float(np.median(vizinho)),
            "minimo_m": float(vizinho.min()),
            "maximo_m": float(vizinho.max()),
        }
    )

distancias_por_classe = pd.DataFrame(linhas_distancia)
distancias_por_classe.to_csv(SAIDA / "distancias_pontos_por_classe.csv", index=False, encoding="utf-8")
distancias_por_classe

## Verificação do GeoPackage

O GeoPackage facilita a inspeção em SIG. Esta etapa é complementar: a reprodução numérica utiliza o Parquet, que preserva os atributos sem depender de drivers geoespaciais. Quando `geopandas` está instalado, o caderno confere a camada agregada e o sistema de referência.

In [ ]:
# Os pontos estão em Parquet; o GeoPackage pode ser gerado quando necessário:
#   import geopandas as gpd
#   gpd.GeoDataFrame(pontos, geometry=gpd.points_from_xy(
#       pontos.longitude, pontos.latitude), crs="EPSG:4326").to_file("points.gpkg")
pontos = pd.read_parquet(raiz / "data" / "points" / "points.parquet")
print(pontos.shape)


## Critério de qualidade das observações

Uma observação de pixel \(i\) na data \(t\) é considerada válida quando há ao menos uma observação clara no composto, a classe SCL não pertence ao conjunto de categorias rejeitadas e as dez reflectâncias são finitas:

\[
Q_{i,t}=\mathbf{1}(\mathrm{CLEAROB}>0)\,
\mathbf{1}(\mathrm{SCL}\notin\{0,1,2,3,8,9,10,11\})
\prod_b\mathbf{1}(\rho_{i,t,b}\text{ finita}).
\]

O produto dos indicadores vale 1 somente quando todas as condições são satisfeitas. Esta máscara é aplicada separadamente em cada combinação `classe × data` antes do PPI.

In [ ]:
def observacoes_validas(tabela, config):
    clearob = pd.to_numeric(tabela["CLEAROB"], errors="coerce")
    scl = pd.to_numeric(tabela["SCL"], errors="coerce")
    reflectancias = tabela[config["bands"]].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    return clearob.gt(0) & ~scl.isin(config["scl_invalid"]) & np.isfinite(reflectancias).all(axis=1)


auditoria_wtss = []
totais_validos_por_data = []
datas_esperadas = int(CONFIG["period"]["expected_dates"])

for codigo in CLASSES:
    caminho = RAIZ / "data" / "wtss" / f"wtss_class_{codigo}.parquet"
    serie = pd.read_parquet(caminho)
    serie["date"] = pd.to_datetime(serie["date"]).dt.strftime("%Y-%m-%d")
    mascara = observacoes_validas(serie, CONFIG)
    por_data = mascara.groupby(serie["date"]).sum().astype(int)

    assert serie["point_id"].nunique() == esperados
    assert serie["date"].nunique() == datas_esperadas
    assert len(serie) == esperados * datas_esperadas

    auditoria_wtss.append(
        {
            "class_code": codigo,
            "rows": len(serie),
            "points": serie["point_id"].nunique(),
            "dates": serie["date"].nunique(),
            "valid_rows": int(mascara.sum()),
            "valid_fraction": float(mascara.mean()),
            "minimum_valid_pixels_per_date": int(por_data.min()),
            "maximum_valid_pixels_per_date": int(por_data.max()),
        }
    )
    totais_validos_por_data.append(
        por_data.rename("valid_pixels").reset_index().assign(class_code=codigo)
    )

auditoria_wtss = pd.DataFrame(auditoria_wtss)
validos_por_data = pd.concat(totais_validos_por_data, ignore_index=True)
auditoria_wtss.to_csv(SAIDA / "auditoria_wtss.csv", index=False, encoding="utf-8")
auditoria_wtss

## Consulta WTSS opcional

O WTSS recebe uma coordenada e devolve a série temporal da coleção. A carga JSON especifica atributos, período, geometria e opções de escala. A função de leitura admite diferentes estruturas de resposta já observadas no serviço e converte o resultado para uma linha por data.

Alterar `EXECUTAR_NOVA_CONSULTA` para `True` realiza uma única consulta demonstrativa. A reconstrução das 5.000 séries requer uma requisição por ponto, cache, retomada e várias horas; por isso não é iniciada automaticamente por este caderno.

In [ ]:
import time
import requests


def carga_wtss(config, ponto):
    return {
        "attributes": [*config["bands"], *config["qa_bands"]],
        "start_date": config["period"]["start"],
        "end_date": config["period"]["end"],
        "geom": {
            "type": "Point",
            "coordinates": [float(ponto["longitude"]), float(ponto["latitude"])],
        },
        "applyAttributeScale": bool(config["wtss"]["apply_attribute_scale"]),
        "pixelCollisionType": "center",
        "masked": bool(config["wtss"]["masked"]),
    }


def requisitar_wtss(config, ponto, cache, forcar=False):
    carga = carga_wtss(config, ponto)
    assinatura = hashlib.sha256(
        json.dumps(carga, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    cache.mkdir(parents=True, exist_ok=True)
    destino = cache / f"{ponto['point_id']}_{assinatura[:16]}.json"
    if destino.exists() and not forcar:
        return json.loads(destino.read_text(encoding="utf-8"))

    url = config["wtss_url"].rstrip("/") + f"/{config['collection_id']}/timeseries"
    ultimo_erro = None
    for tentativa in range(1, int(config["wtss"]["retries"]) + 1):
        try:
            resposta = requests.post(
                url,
                json=carga,
                timeout=float(config["wtss"]["timeout_seconds"]),
            )
            resposta.raise_for_status()
            resultado = resposta.json()
            if not resultado.get("results"):
                raise ValueError("Resposta WTSS sem resultados.")
            temporario = destino.with_suffix(".tmp")
            temporario.write_text(json.dumps(resultado, ensure_ascii=False), encoding="utf-8")
            temporario.replace(destino)
            return resultado
        except Exception as erro:
            ultimo_erro = erro
            if tentativa < int(config["wtss"]["retries"]):
                time.sleep(float(config["wtss"]["retry_delay_seconds"]) * 2 ** (tentativa - 1))
    raise RuntimeError(f"Falha WTSS: {ultimo_erro}")


def converter_resposta_wtss(resultado, ponto, config):
    bloco = resultado["results"][0]
    serie = bloco.get("time_series") or bloco.get("timeseries")
    datas = bloco.get("timeline") or bloco.get("dates")
    atributos = bloco.get("attributes") or [*config["bands"], *config["qa_bands"]]
    linhas = []

    if isinstance(serie, dict):
        datas = serie.get("timeline") or datas
        valores = serie.get("values")
        if not datas or not isinstance(valores, dict):
            raise ValueError("Resposta WTSS sem timeline e valores.")
        for indice, data in enumerate(datas):
            linhas.append(
                {"date": str(data)[:10], **{nome: valores.get(nome, [None] * len(datas))[indice] for nome in atributos}}
            )
    elif isinstance(serie, list) and serie and isinstance(serie[0], dict):
        for item in serie:
            valores = item.get("values", item)
            data = item.get("date") or item.get("datetime")
            linhas.append({"date": str(data)[:10], **{nome: valores.get(nome) for nome in atributos}})
    elif isinstance(serie, list) and datas and len(serie) == len(datas):
        for data, valores in zip(datas, serie):
            linhas.append({"date": str(data)[:10], **dict(zip(atributos, valores))})
    elif isinstance(serie, list) and serie and isinstance(serie[0], list):
        for valores in serie:
            linhas.append({"date": str(valores[0])[:10], **dict(zip(atributos, valores[1:]))})
    else:
        raise ValueError("Estrutura da resposta WTSS não reconhecida.")

    tabela = pd.DataFrame(linhas)
    tabela = tabela[tabela["date"].between(config["period"]["start"], config["period"]["end"])].copy()
    tabela.insert(0, "point_id", ponto["point_id"])
    tabela.insert(1, "class_code", str(ponto["class_code"]).zfill(2))
    tabela["systematic_order"] = int(ponto["systematic_order"])
    return tabela


EXECUTAR_NOVA_CONSULTA = False

if EXECUTAR_NOVA_CONSULTA:
    ponto_exemplo = pontos.sort_values(["class_code", "systematic_order"]).iloc[0]
    resposta = requisitar_wtss(CONFIG, ponto_exemplo, SAIDA / "wtss_cache")
    exemplo_wtss = converter_resposta_wtss(resposta, ponto_exemplo, CONFIG)
    display(exemplo_wtss.head())
else:
    print("Consulta remota desativada; foram auditadas as séries publicadas em data/wtss/.")

## Resultado desta etapa

A etapa é aceita quando existem exatamente 5.000 pixels sem duplicação dentro das classes, 1.000 em cada classe, uma única seed de amostragem e 184 registros por pixel. As tabelas de auditoria são gravadas em `outputs/standalone/` e alimentam a verificação do próximo caderno.

In [ ]:
resultado_auditoria = {
    "points": int(len(pontos)),
    "points_per_class": {codigo: int(valor) for codigo, valor in contagens.items()},
    "duplicate_pixels": int(duplicados),
    "sampling_seeds": seeds,
    "wtss_rows": int(auditoria_wtss["rows"].sum()),
    "dates_per_point": datas_esperadas,
    "status": "PASS",
}
(SAIDA / "auditoria_entrada.json").write_text(
    json.dumps(resultado_auditoria, ensure_ascii=False, indent=2), encoding="utf-8"
)
resultado_auditoria